# Homework 5 — UFO sightings: When and where do people report them?

This notebook creates two Altair visualizations from the UFO dataset in IS 445 Homework 5. The first explores reports by month and hour. The second uses a dropdown to compare reported shapes across US states.

Data: https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/ufo-scrubbed-geocoded-time-standardized-00.csv

These are self-reported sightings, not verified UFO events.

In [ ]:
import pandas as pd
import altair as alt
from pathlib import Path

DATA_URL = "https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/ufo-scrubbed-geocoded-time-standardized-00.csv"

columns = [
    "date_sighted", "city", "state", "country", "shape",
    "duration_seconds", "duration_reported", "notes",
    "date_reported", "latitude", "longitude"
]

# Read the data directly from the course URL.
df = pd.read_csv(DATA_URL, names=columns, header=None,
                 dtype=str, on_bad_lines="skip", low_memory=False)
df = df[~df["date_sighted"].str.lower().isin(["datetime", "date_sighted"])].copy()

print("Records read:", len(df))
df.head()

## Preparing the data

The date strings contain both calendar dates and times. I extract month and hour to study timing patterns. I also standardize the state and shape labels to make the second plot easier to filter.

In [ ]:
parts = df["date_sighted"].str.extract(
    r"^\s*(\d{1,2})/(\d{1,2})/(\d{4})\s+(\d{1,2}):"
)
df["month"] = pd.to_numeric(parts[0], errors="coerce")
df["year"] = pd.to_numeric(parts[2], errors="coerce")
df["hour"] = pd.to_numeric(parts[3], errors="coerce")
df["country"] = df["country"].str.strip().str.lower()
df["state"] = df["state"].str.strip().str.lower()
df["shape"] = df["shape"].str.strip().str.lower()

df[["month", "hour", "state", "shape"]].head()

## Visualization 1 — When do reports occur?

The heatmap groups reports by month and hour. Each cell has the number of reports in that month/hour combination.

In [ ]:
valid_dates = df[
    df["month"].between(1, 12) & df["hour"].between(0, 23)
]

by_time = (valid_dates.groupby(["month", "hour"])
           .size().reset_index(name="Reports"))

all_times = pd.MultiIndex.from_product(
    [range(1, 13), range(24)], names=["month", "hour"]
)
by_time = (by_time.set_index(["month", "hour"])
           .reindex(all_times, fill_value=0).reset_index())

chart1 = alt.Chart(by_time).mark_rect().encode(
    x=alt.X("hour:O", title="Hour of day", sort=list(range(24))),
    y=alt.Y("month:O", title="Month", sort=list(range(1, 13))),
    color=alt.Color("Reports:Q", scale=alt.Scale(scheme="viridis"),
                    title="Number of reports"),
    tooltip=[alt.Tooltip("month:O", title="Month"),
             alt.Tooltip("hour:O", title="Hour"),
             alt.Tooltip("Reports:Q", title="Reports")]
).properties(title="UFO reports by month and hour", width=680, height=330)

chart1

## Visualization 2 — Reported shapes across US states

I keep US reports with two-letter state codes, then use the fifteen most frequently reported states and ten most common shapes. The dropdown switches the shape while keeping the same states in view.

In [ ]:
us_reports = df[
    (df["country"] == "us") &
    (df["state"].str.fullmatch(r"[a-z]{2}", na=False)) &
    (df["shape"].notna()) &
    (df["shape"] != "")
].copy()

popular_shapes = us_reports["shape"].value_counts().head(10).index.tolist()
popular_states = us_reports["state"].value_counts().head(15).index.tolist()

state_shape = (us_reports[
    us_reports["shape"].isin(popular_shapes) &
    us_reports["state"].isin(popular_states)
].groupby(["state", "shape"]).size().reset_index(name="Reports"))

all_pairs = pd.MultiIndex.from_product(
    [popular_states, popular_shapes], names=["state", "shape"]
)
state_shape = (state_shape.set_index(["state", "shape"])
               .reindex(all_pairs, fill_value=0).reset_index())
state_shape["state"] = state_shape["state"].str.upper()

print("Shapes:", popular_shapes)
print("States included:", len(popular_states))
state_shape.head()

In [ ]:
dropdown = alt.binding_select(options=popular_shapes, name="Reported shape: ")
selected_shape = alt.param(
    name="SelectedShape", value=popular_shapes[0], bind=dropdown
)

chart2 = (alt.Chart(state_shape)
    .mark_bar(cornerRadiusEnd=3)
    .encode(
        x=alt.X("Reports:Q", title="Number of reports"),
        y=alt.Y("state:N", title="US state", sort=[s.upper() for s in popular_states]),
        color=alt.Color("Reports:Q", scale=alt.Scale(scheme="blues"),
                        legend=None),
        tooltip=[alt.Tooltip("state:N", title="State"),
                 alt.Tooltip("shape:N", title="Reported shape"),
                 alt.Tooltip("Reports:Q", title="Reports")]
    )
    .add_params(selected_shape)
    .transform_filter("datum.shape == SelectedShape")
    .properties(title="UFO reports by state and reported shape", width=630, height=440)
)

chart2

## Exporting to the Jekyll website

This last cell saves the two chart specifications in the GitHub Pages repository's assets/json/ folder. After running the notebook, commit and push the JSON files so the charts appear on the project page.

In [ ]:
repo = Path.cwd()
if repo.name == "python_notebooks":
    repo = repo.parent

output = repo / "assets" / "json"
output.mkdir(parents=True, exist_ok=True)

chart1.save(str(output / "hw5_ufo_hour_month.json"))
chart2.save(str(output / "hw5_ufo_shape_states.json"))

print("Saved:", output / "hw5_ufo_hour_month.json")
print("Saved:", output / "hw5_ufo_shape_states.json")